# Notebook 13 — Two-Parameter Schedule Shaping

This notebook extends the phase-locked adiabatic optimization by giving the control schedules **shape parameters**.

Instead of using one fixed pulse shape, we let the optimizer tune:

- total gate time $T$,
- detuning scale factor $\alpha$,
- maximum drive $\Omega_{max}$,
- initial detuning offset $\Delta_0$,
- pulse-shape exponent $p$ for $\Omega(t)$,
- sweep-shape exponent $q$ for $\Delta(t)$.

The idea is simple:

- $p$ controls how sharply the drive turns on and off,
- $q$ controls where detuning accumulates phase during the sweep.

This gives the optimizer a way to align:
1. low leakage,
2. population return,
3. entangling phase near $\pi$.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qutip import basis, qeye, tensor, sigmax, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Shape-controlled schedules

In [ ]:
def omega_schedule(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_schedule(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))


## Time-dependent Hamiltonian

In [ ]:
H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_schedule(t, T=T, omega_max=omega_max, p=p)

    def coeff_delta(t, args=None):
        return delta_schedule(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)

    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Effective-unitary helpers

In [ ]:
def run_shaped_evolution(psi0, T, omega_max, alpha, V, delta0, p, q, n_steps=340):
    H = build_time_dependent_hamiltonian(
        T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=delta0, p=p, q=q
    )
    times = np.linspace(0.0, T, n_steps)
    result = sesolve(H, psi0, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(T, omega_max, alpha, V, delta0, p, q):
    columns = []
    for psi0 in basis_states:
        psi_final = run_shaped_evolution(
            psi0, T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=delta0, p=p, q=q
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate metrics

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, U_target=phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))


## Phase-locked score

In [ ]:
V = 4.0

def improved_score(U_eff):
    pc = phase_corrected_fidelity(U_eff)
    leak = leakage_metric(U_eff)
    phi = entangling_phase(U_eff)
    phase_term = np.cos(phi - np.pi)
    return float(pc * (0.5 + 0.5 * phase_term) - 0.30 * leak)

def objective(x):
    T, alpha, omega_max, delta0, p, q = x
    if T <= 6.0 or alpha <= 0.10 or omega_max <= 0.20 or p <= 0.5 or q <= 0.5:
        return 1e6
    if delta0 < -0.5 or delta0 > 2.5 or alpha > 1.5 or omega_max > 2.0 or p > 3.0 or q > 3.0:
        return 1e6

    U_eff = build_effective_unitary(
        T=float(T),
        omega_max=float(omega_max),
        alpha=float(alpha),
        V=V,
        delta0=float(delta0),
        p=float(p),
        q=float(q),
    )
    return -improved_score(U_eff)


## Baseline shaped candidate

In [ ]:
x0 = np.array([26.0, 0.30, 1.10, 0.75, 1.0, 1.0])  # [T, alpha, omega_max, delta0, p, q]

U0 = build_effective_unitary(
    T=x0[0], omega_max=x0[2], alpha=x0[1], V=V, delta0=x0[3], p=x0[4], q=x0[5]
)

print('Baseline shaped candidate:')
print(f'  T = {x0[0]:.4f}')
print(f'  alpha = {x0[1]:.4f}')
print(f'  omega_max = {x0[2]:.4f}')
print(f'  delta0 = {x0[3]:.4f}')
print(f'  p = {x0[4]:.4f}')
print(f'  q = {x0[5]:.4f}')
print(f'  raw fidelity:             {process_fidelity(U0):.6f}')
print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U0):.6f}')
print(f'  entangling phase / pi:    {entangling_phase(U0)/np.pi:.6f}')
print(f'  leakage:                  {leakage_metric(U0):.6f}')
print(f'  improved score:           {improved_score(U0):.6f}')


## Optimization with shape parameters

In [ ]:
bounds = [
    (10.0, 45.0),  # T
    (0.10, 1.20),  # alpha
    (0.40, 1.80),  # omega_max
    (-0.50, 2.50), # delta0
    (0.60, 2.50),  # p
    (0.60, 2.50),  # q
]

result = minimize(
    objective,
    x0=x0,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 50}
)

x_opt = result.x
U_opt = build_effective_unitary(
    T=x_opt[0], omega_max=x_opt[2], alpha=x_opt[1], V=V,
    delta0=x_opt[3], p=x_opt[4], q=x_opt[5]
)

print('Optimization success:', result.success)
print('Message:', result.message)
print()
print('Optimized shaped candidate:')
print(f'  T = {x_opt[0]:.6f}')
print(f'  alpha = {x_opt[1]:.6f}')
print(f'  omega_max = {x_opt[2]:.6f}')
print(f'  delta0 = {x_opt[3]:.6f}')
print(f'  p = {x_opt[4]:.6f}')
print(f'  q = {x_opt[5]:.6f}')
print(f'  raw fidelity:             {process_fidelity(U_opt):.6f}')
print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_opt):.6f}')
print(f'  entangling phase / pi:    {entangling_phase(U_opt)/np.pi:.6f}')
print(f'  leakage:                  {leakage_metric(U_opt):.6f}')
print(f'  improved score:           {improved_score(U_opt):.6f}')


## Effective unitary before vs after shaping optimization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

im0 = axes[0].imshow(np.abs(U0), origin='lower', aspect='equal')
axes[0].set_title(r'$|U_{eff}|$ baseline shaped')
axes[0].set_xticks(range(4), basis_labels)
axes[0].set_yticks(range(4), basis_labels)

im1 = axes[1].imshow(np.abs(U_opt), origin='lower', aspect='equal')
axes[1].set_title(r'$|U_{eff}|$ optimized shaped')
axes[1].set_xticks(range(4), basis_labels)
axes[1].set_yticks(range(4), basis_labels)

fig.colorbar(im1, ax=axes.ravel().tolist(), label='magnitude')
plt.tight_layout()
plt.show()


## Score slices around optimized shape parameters

In [ ]:
T_vals = np.linspace(max(10.0, x_opt[0]-10), min(45.0, x_opt[0]+10), 36)
alpha_vals = np.linspace(max(0.10, x_opt[1]-0.35), min(1.20, x_opt[1]+0.35), 36)
omega_vals = np.linspace(max(0.40, x_opt[2]-0.50), min(1.80, x_opt[2]+0.50), 36)
delta0_vals = np.linspace(max(-0.50, x_opt[3]-0.80), min(2.50, x_opt[3]+0.80), 36)
p_vals = np.linspace(max(0.60, x_opt[4]-0.80), min(2.50, x_opt[4]+0.80), 36)
q_vals = np.linspace(max(0.60, x_opt[5]-0.80), min(2.50, x_opt[5]+0.80), 36)

def score_at(T, alpha, omega_max, delta0, p, q):
    U = build_effective_unitary(T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=delta0, p=p, q=q)
    return improved_score(U)

score_T = [score_at(T, x_opt[1], x_opt[2], x_opt[3], x_opt[4], x_opt[5]) for T in T_vals]
score_alpha = [score_at(x_opt[0], a, x_opt[2], x_opt[3], x_opt[4], x_opt[5]) for a in alpha_vals]
score_omega = [score_at(x_opt[0], x_opt[1], om, x_opt[3], x_opt[4], x_opt[5]) for om in omega_vals]
score_delta0 = [score_at(x_opt[0], x_opt[1], x_opt[2], d0, x_opt[4], x_opt[5]) for d0 in delta0_vals]
score_p = [score_at(x_opt[0], x_opt[1], x_opt[2], x_opt[3], p, x_opt[5]) for p in p_vals]
score_q = [score_at(x_opt[0], x_opt[1], x_opt[2], x_opt[3], x_opt[4], q) for q in q_vals]


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 10))

axes[0,0].plot(T_vals, score_T); axes[0,0].axvline(x_opt[0], linestyle='--'); axes[0,0].set_title('Score vs T')
axes[0,1].plot(alpha_vals, score_alpha); axes[0,1].axvline(x_opt[1], linestyle='--'); axes[0,1].set_title('Score vs alpha')
axes[1,0].plot(omega_vals, score_omega); axes[1,0].axvline(x_opt[2], linestyle='--'); axes[1,0].set_title('Score vs omega_max')
axes[1,1].plot(delta0_vals, score_delta0); axes[1,1].axvline(x_opt[3], linestyle='--'); axes[1,1].set_title('Score vs delta0')
axes[2,0].plot(p_vals, score_p); axes[2,0].axvline(x_opt[4], linestyle='--'); axes[2,0].set_title('Score vs p')
axes[2,1].plot(q_vals, score_q); axes[2,1].axvline(x_opt[5], linestyle='--'); axes[2,1].set_title('Score vs q')

for ax in axes.ravel():
    ax.set_ylabel('Improved score')

plt.tight_layout()
plt.show()


## Local 2D map over $(T, \alpha)$ with optimized shapes

In [ ]:
T_local = np.linspace(max(10.0, x_opt[0]-10), min(45.0, x_opt[0]+10), 28)
alpha_local = np.linspace(max(0.10, x_opt[1]-0.30), min(1.20, x_opt[1]+0.30), 24)

score_map = np.zeros((len(alpha_local), len(T_local)))
pc_map = np.zeros((len(alpha_local), len(T_local)))
phi_map = np.zeros((len(alpha_local), len(T_local)))
leak_map = np.zeros((len(alpha_local), len(T_local)))

for i, a in enumerate(alpha_local):
    for j, T in enumerate(T_local):
        U = build_effective_unitary(
            T=T, omega_max=x_opt[2], alpha=a, V=V,
            delta0=x_opt[3], p=x_opt[4], q=x_opt[5]
        )
        score_map[i, j] = improved_score(U)
        pc_map[i, j] = phase_corrected_fidelity(U)
        phi_map[i, j] = entangling_phase(U) / np.pi
        leak_map[i, j] = leakage_metric(U)


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(score_map, origin='lower', aspect='auto', extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]])
plt.contour(T_local, alpha_local, score_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Improved score over local $(T, \alpha)$ with optimized shapes')
plt.colorbar(im, label='improved score')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_map, origin='lower', aspect='auto', extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]])
plt.contour(T_local, alpha_local, pc_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Phase-corrected fidelity over local $(T, \alpha)$ with optimized shapes')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(phi_map, origin='lower', aspect='auto', extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]], vmin=-1, vmax=1)
plt.contour(T_local, alpha_local, phi_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Entangling phase / $\pi$ over local $(T, \alpha)$ with optimized shapes')
plt.colorbar(im, label=r'$\phi_{ent}/\pi$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(leak_map, origin='lower', aspect='auto', extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]])
plt.contour(T_local, alpha_local, leak_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Leakage over local $(T, \alpha)$ with optimized shapes')
plt.colorbar(im, label='leakage')
plt.tight_layout()
plt.show()


## Interpretation

This notebook adds the missing schedule-shaping degrees of freedom.

The new ingredients are:
- $p$: controls how sharply the drive turns on and off,
- $q$: controls where detuning accumulates phase during the sweep.

These shape parameters let the optimizer align:
- low leakage,
- population return,
- phase locking,

instead of trying to force all three with only $(T, \alpha, \Omega_{max}, \Delta_0)$.


## Optional next steps

Natural next upgrades are:

- compare the best shaped adiabatic candidate against all earlier notebooks in one final summary notebook,
- add open-system noise to the optimized shaped protocol,
- try richer schedules such as two-segment or spline parameterizations,
- switch from local optimization to multi-start optimization for stronger robustness.
